# သင်ခန်းစာ ၂ - Microsoft Agent Framework ကို စူးစမ်းလေ့လာခြင်း

**Microsoft Agent Framework (MAF)** သည် AI စက်ရုပ်များ တည်ဆောက်ရာတွင် ချိတ်ဆက်သုံးစွဲနိုင်သော စနစ်တစ်ခု ဖြစ်သည်။ ၎င်းသည် သန့်ရှင်းပြီး ပေါင်းစပ်နိုင်သော ဖွဲ့စည်းမှုကို အဓိက အစိတ်အပိုင်း ၄ မျိုးဖြင့် ပံ့ပိုးပေးသည်။

- **Client** – AI မော်ဒယ် endpoint နှင့် ချိတ်ဆက်ကာ ဆက်သွယ်မှုကို ကိုင်တွယ်သည်
- **Agent** – client ကို အညွှန်းများနှင့် ကိရိယာ သတ်မှတ်ချက်များဖြင့် ပတ်ပတ်လည် ထုပ်ပိုးပေးသည်
- **Tools** – မော်ဒယ်မှ ခေါ်ယူနိုင်သော စိတ်ကြိုက် လုပ်ဆောင်ချက်များဖြင့် agent ၏ စွမ်းဆောင်ရည်များကို တိုးချဲ့ပေးသည်
- **Session** – 多-အကြိမ် စကားပြောဆိုခြင်းများအတွက် စကားပုံမှတ်တမ်းကို ထိန်းသိမ်းထားသည်

ဤသင်ခန်းစာတွင် ကျွန်ုပ်တို့သည် မျဉ်းပေါ်လွင့္ခရီးစဉ်ကို စစ်ဆေးနိုင်သော **ခရီးသွားစာရင်း agent** တစ်ခုကို ဖန်တီးမည် ဖြစ်သည်။


## စတင်တပ်ဆင်ခြင်း


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Agent Framework ဖွဲ့စည်းမှုကိုနားလည်ခြင်း

Microsoft Agent Framework သည် အလွှာသွားအဆင့်များဖြင့် ဖွဲ့စည်းထားသည်။

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – `FoundryChatClient` သည် Azure OpenAI တပ်ဆင်မှုသို့ ချိတ်ဆက်သည်။ ၎င်းသည် မှတ်ပုံတင်ခြင်း၊ တောင်းဆိုမှုဖောင်မတ်လုပ်ခြင်းနှင့် ဖြေဆိုချက်ခွဲခြမ်းစိတ်ဖြာခြင်းတို့ကို ကိုယ်စားပြုသည်။
2. **Agent** – `provider.create_agent()` မှတဆင့် client မှ ဖန်တီးပြီး၊ agent သည် မော်ဒယ် ဝင်ရောက်မှုအား စနစ်ညွှန်းချက်များနှင့် ကိရိယာများဖြင့်ပေါင်းစပ်ပေးသည်။
3. **Tools** – `@tool` ဖြင့် အမှတ်အသားပြုထားသော Python function များဖြစ်ပြီး agent သည် လုပ်ဆောင်ချက်များပြုလုပ်ရန် သို့မဟုတ် ဒေတာ ရယူရန် အသုံးပြုနိုင်သည်။
4. **Session** – `agent.create_session()` မှတဆင့် ဖန်တီးသော `AgentSession` အရာဝတ္ထုဖြစ်ပြီး စကားပြောမှု အတိတ်မှတ်ဉာဏ်များကို သိမ်းဆည်းကာ multi-turn စကားဝိုင်းတွင် agent သည် ယခင်ပတ်ဝန်းကျင်ကို မှတ်မိနိုင်သည်။

အလွှာတိုင်းကို တစ်ဆင့်စီ လိုက်လံတည်ဆောက်ကြစို့။


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## @tool Decorator ဖြင့် ကိရိယာများ ထည့်သွင်းခြင်း

ကိရိယာများက ကိုယ်စားလှယ်များအား စာသားနှင့်သာမက အခြား လုပ်ဆောင်ချက်များကိုလည်း ပြုလုပ်နိုင်စေသည်။ `@tool` decorator သည် ပုံမှန် Python function တစ်ခုကို ကိုယ်စားလှယ်သည် ခေါ်ယူနိုင်သော အရာတစ်ခုအဖြစ် ပြောင်းလဲပေးသည်။

အဓိကအချက်များ -
- `Annotated[type, "description"]` ကို အသုံးပြု၍ ပုံစံသည် အချက်တစ်ခုချင်းစီကို နားလည်စေရမည်။
- docstring သည် ပုံစံမြင် မည့် ကိရိယာဖော်ပြချက် ဖြစ်သည်။
- `approval_mode="never_require"` သည် အသုံးပြုသူ အတည်ပြုမှုမလိုအပ်ပဲ ကိရိယာကို အလိုအလျောက် လည်ပတ်စေသည်။


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## စက်ရုပ်တစ်ခုကို ကိရိယာများဖြင့် ဖန်တီးခြင်း

ယခု၌ client၊ ညွှန်ကြားချက်များ၊ နှင့် ကိရိယာများကို စက်ရုပ်အဖြစ် ပေါင်းစပ်ပါမည်။ `instructions` များသည် system prompt အဖြစ် လုပ်ဆောင်ပြီး မည်သည့် persona နှင့် အပြုအမှုအပေါ် သတ်မှတ်ချက်ထားသည်ကို ဖော်ပြပါသည်။


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## အကြိမ်ခွဲစကားပြောဆိုမှုများနှင့် Sessions  

`agent.create_session()` ဖြင့် ဖန်တီးထားသော `AgentSession` သည် စကားပြောဆိုမှုတစ်ခုလုံးရှိ မက်ဆေ့ချ်များအားလုံးကို စောင့်ကြည့်ထားသည်။ တစ်ခုချင်း `agent.run()` ခေါ်သောအခါ တူညီသည့် session ကိုပေးပို့ခြင်းဖြင့် agent သည် စကားပြောဆိုမှု စာရင်း အပြည့်အစုံသို့ ဝင်ရောက်နိုင်ကာ အရင်က မက်ဆေ့ချ်များကို ပြန်လည်ညွှန်ပြနိုင်သည်။  

အမြဲတမ်း turn တစ်ခုစီတွင် availability checker ကို ခေါ်ယူနိုင်ဖို့အတွက် `tools=[check_destination_availability]` ကို ပေးပို့ထားသည်။  


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## စာချုပ်ချက်

ဒီသင်ခန်းစာမှာ Microsoft Agent Framework ရဲ့ အခြေခံဖွဲ့စည်းမှု ၄ ခုကို သင်လေ့လာခဲ့တယ်။

| အကြောင်းအရာ | သင်လေ့လာခဲ့သည်များ |
|---------|------------------|
| **Client** | `FoundryChatClient` သည် credential-based auth ဖြင့် Azure OpenAI ကို ချိတ်ဆက်သည် |
| **Agent** | `provider.create_agent()` သည် မော်ဒယ်ချိတ်ဆက်မှု၊ အမိန့်များနှင့် နာမည်တို့ကို ပေါင်းစပ်ပေးသည် |
| **Tools** | `@tool` decorator သည် Python function များကို agent အတွက် ခေါ်ရန် ဖော်ပြပေးသည် |
| **Session** | `agent.create_session()` သည် စကားပြော သိမ်းဆည်းမှုကို turn များစဉ်ဆက်လက် ထိန်းသိမ်းထားသည် |

အဲဒီဖွဲ့စည်းမှုအစိတ်အပိုင်းများသည် သဘာဝစကားပြောဆိုမှုကို ထိန်းသိမ်းနိုင်ပြီး ပြင်ပ function များ ခေါ်ယူနိုင်ခြင်းနှင့် အကြောင်းအရာဆက်လက်ထိန်းသိမ်းခြင်းတို့ဖြင့် agent များကို ဖန်တီးပေးသည် — နောက်ပိုင်းသင်ခန်းစာများတွင် ပိုမိုတိုးတက်သော agentic ပုံစံများအတွက် မူလအခြေခံဖြစ်သည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
